In [114]:
import json 

import os 

In [115]:
import requests

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

In [116]:
documents_raw[2]

{'course': 'mlops-zoomcamp',
 'documents': [{'text': 'MLOps Zoomcamp FAQ\nThe purpose of this document is to capture frequently asked technical questions.\nWe did this for our data engineering course, and it worked quite well. Check this document for inspiration on how to structure your questions and answers:\nData Engineering Zoomcamp FAQ\n[Problem description]\n[Solution description]\n(optional) Added by Name',
   'section': '+-General course questions',
   'question': 'Format for questions: [Problem title]'},
  {'text': 'Approximately 3 months. For each module, about 1 week with possible deadline extensions (in total 6~9 weeks), 2 weeks for working on the capstone project and 1 week for peer review.',
   'section': '+-General course questions',
   'question': 'What is the expected duration of this course or that for each module?'},
  {'text': 'The difference is the Orchestration and Monitoring modules. Those videos will be re-recorded. The rest should mostly be the same.\nAlso all o

In [117]:
documents = []

for course_dict in documents_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [118]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [119]:
import minsearch
from openai import OpenAI
import json 
from elasticsearch import Elasticsearch

import os 
from groq import Groq 
from dotenv import load_dotenv 
load_dotenv()

client = Groq(api_key = os.environ.get('GROQ_API_KEY'))

In [120]:
def search(query):
    boost = {'question': 3.0 , 'section' : 0.5}

    results = index.search(
        query = query,
        boost_dict = boost,
        num_results = 5
    )

    return results 

In [121]:
def build_prompt(query, results):
    prompt_template = """ 
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.


QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context = ""

    for doc in results:
        if isinstance(doc, dict):
                payload = doc   #safely access metadata
        else:
             payload = getattr(doc, "payload", {})

        context += f"section: {payload.get('section', '')}\n"
        context += f"question: {payload.get('question', '')}\n"
        context += f"answer: {payload.get('text', '')}\n\n"

        #context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    
    return prompt 

In [122]:
def llm(prompt):
    response = client.chat.completions.create(
        model= 'llama3-70b-8192',
        messages= [{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [123]:



def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [124]:
rag('how do I run kafka?')

'Based on the FAQ database, to run Kafka, you need to follow these steps:\n\n**For Java Kafka:**\nIn the project directory, run:\n`java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java`\n\n**For Python Kafka:**\n1. Create a virtual environment and install requirements.txt:\n`python -m venv env`\n`source env/bin/activate`\n`pip install -r ../requirements.txt`\n2. Activate the virtual environment:\n`source env/bin/activate` (on MacOS, Linux) or `env\\Scripts\\activate` (on Windows)\n3. Run the Python file in the virtual environment.\n\nNote: Make sure to create the virtual environment only for running the Python file, and ensure that Docker images are up and running.'

In [125]:
rag('the course has already started, can I still enroll?')

'According to the FAQ database, the answer to the question "Can I still enroll in the course even though it has already started?" is:\n\n"Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course. You can still be eligible for a certificate if you submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline."'

RAG with Vector Search

In [126]:
from qdrant_client import QdrantClient, models

qd_client = QdrantClient("http://localhost:6333")
qd_client.get_collections()

/var/folders/kp/_4404mkx1zn_85zd51sfg3440000gn/T/ipykernel_23318/1342208640.py:3: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.9.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qd_client = QdrantClient("http://localhost:6333")


CollectionsResponse(collections=[CollectionDescription(name='zoomcamp-rag')])

In [127]:
from fastembed.embedding import DefaultEmbedding

In [128]:
# Load the model from FastEmbed
model_handle = DefaultEmbedding(model_name="jinaai/jina-embeddings-v2-small-en")

# Confirm dimensions (usually 512 for small models)
EMBEDDING_DIMENSIONALITY = 512
collection_name = "zoomcamp-faq"

In [130]:


# Create the collection with specified sparse vector parameters
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config= models.VectorParams(
        size= EMBEDDING_DIMENSIONALITY, #Dimesionality of the vectors
        distance= models.Distance.COSINE  # Distance metric for similarity search
    )
)

True

In [131]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [132]:
points = []
id = 0


for course_dict in documents_raw:
    for doc in course_dict['documents']:
        text = doc['text']

        # Generate embedding
        vector = list(model_handle.embed(documents=[text]))[0]   # FastEmbed returns a list


        point = models.PointStruct(
            id = id,
            #vector = models.Document(text= doc['text'], model= model_handle) ,   #embed text locally with "" from FastEmbed
            vector=vector,
            payload={
                "text": text,
                "section": doc['section'],
                "course": course_dict['course']

            }   # save all needed metadata fields
        
        )
        points.append(point)

        id += 1

In [133]:
qd_client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [134]:
question = 'I just discovered the course. Can I still join it?'

In [149]:
def search(query, limit=1):
    
    # Manually embed the query using FastEmbed
    query_vector = list(model_handle.embed(documents=[query]))[0]

    # Perform the search using the raw vector
    results = qd_client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit= 5,
        with_payload=True
    )

    return results

In [150]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [151]:
rag('how do I run kafka?')

/var/folders/kp/_4404mkx1zn_85zd51sfg3440000gn/T/ipykernel_23318/1838244662.py:7: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qd_client.search(


'To run Kafka, you need to start the Kafka broker docker container. You can do this by running `docker compose up -d` in the docker compose yaml file folder. Make sure the container is running by using `docker ps` to confirm. Additionally, ensure that you have set the correct server URL in the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` and updated the cluster key and secrets in `Secrets.java`.'

In [153]:
results = rag('how do I run kafka?')

print(results)

/var/folders/kp/_4404mkx1zn_85zd51sfg3440000gn/T/ipykernel_23318/1838244662.py:7: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qd_client.search(


Based on the provided context, to run Kafka, you should:

1. Make sure the Kafka broker docker container is running. You can check this using `docker ps`. If it's not running, navigate to the docker compose yaml file folder and run `docker compose up -d` to start all the instances.

Note that there is no specific command provided in the context to run Kafka directly. The solution is focused on resolving issues that may prevent Kafka from running correctly.
